# Get Conservation and ENCODE from UCSC

In [7]:
import json, urllib.request
from pathlib import Path

PROJECT_ROOT = Path("../").resolve()
OUT_DIR = PROJECT_ROOT / "tracks" / "ucsc_raw"

GENOME = "hg38"
chrom, start1, end1 = "chr19", 11_039_463, 11_163_820   # 1-based inclusive, like the browser

In [8]:
import json, urllib.request

def ucsc_items(track, chrom, start1, end1, genome="hg38"):
    """Items of a UCSC track overlapping a 1-based inclusive region."""
    url = (f"https://api.genome.ucsc.edu/getData/track?genome={genome};track={track}"
           f";chrom={chrom};start={start1 - 1};end={end1}")
    data = json.load(urllib.request.urlopen(url))[track]
    return data[chrom] if isinstance(data, dict) else data   # signal tracks are keyed by chrom

def write_bedgraph(items, chrom, path, name):
    lines = [f'track type=bedGraph name="{name}"']
    lines += [f"{chrom}\t{it['start']}\t{it['end']}\t{it['value']}" for it in items]
    path.write_text("\n".join(lines) + "\n", newline="\n")

def write_bed(items, chrom, path, name):
    lines = [f'track name="{name}" itemRgb="On"']
    for it in items:
        s, e = it["chromStart"], it["chromEnd"]
        rgb = it.get("reserved") or it.get("itemRgb") or "0"
        lines.append(f"{chrom}\t{s}\t{e}\t{it.get('name', '')}\t0\t.\t{s}\t{e}\t{rgb}")
    path.write_text("\n".join(lines) + "\n", newline="\n")

# e.g. (chrom, start1, end1, OUT_DIR from your earlier cells)
write_bedgraph(ucsc_items("phastCons100way", chrom, start1, end1), chrom,
               OUT_DIR / "phastcons100.bedGraph", "PhastCons100")
write_bed(ucsc_items("encodeCcreCombined", chrom, start1, end1), chrom,
          OUT_DIR / "encode3_ccre.bed", "ENCODE3")
write_bedgraph(ucsc_items("phastCons30way", chrom, start1, end1), chrom,
               OUT_DIR / "phastcons30.bedGraph", "PhastCons30")
write_bed(ucsc_items("cCREregistry", chrom, start1, end1), chrom,
          OUT_DIR / "encode4_ccre.bed", "ENCODE4")

In [9]:
for it in ucsc_items("cCREregistry", chrom, start1, end1):
    if (it.get("reserved") or it.get("itemRgb")) == "216,118,236":
        print(it)
        break

{'chrom': 'chr19', 'chromStart': 11066898, 'chromEnd': 11067168, 'name': 'EH38E4192819', 'score': 0, 'strand': '.', 'thickStart': 11066898, 'thickEnd': 11067168, 'reserved': '216,118,236', 'cCRE_class': 'TF', 'DNase_maxZ': 1.49, 'H3K4me3_maxZ': 1.54, 'H3K27ac_maxZ': 2.7, 'CTCF_maxZ': 1.57}


In [ ]:
# Class code → plain-language meaning
CCRE_CLASS_TEXT = {
    "PLS": "promoter-like",
    "pELS": "proximal enhancer-like (<2 kb from TSS)",
    "dELS": "distal enhancer-like (>2 kb from TSS)",
    "DNase-H3K4me3": "H3K4me3 away from any TSS",
    "CA-H3K4me3": "H3K4me3 away from any TSS",
    "CTCF-only": "CTCF-bound, no promoter/enhancer marks",
    "CA-CTCF": "CTCF-bound, no promoter/enhancer marks",
    "CA-TF": "accessible + TF-bound",
    "CA": "accessible only",
    "CA-only": "accessible only",
    "TF": "TF-bound only",
}
# Color → class code, for tracks without a class field
CCRE_COLOR_CLASS = {
    "255,0,0": "PLS",
    "255,167,0": "pELS",
    "255,205,0": "dELS",
    "255,170,170": "DNase-H3K4me3",
    "0,176,240": "CTCF-only",
    "216,118,236": "TF",
}
CLASS_FIELDS = ("cCRE_class", "encodeLabel", "ucscLabel", "label")
unmapped_colors = set()

# UCSC's short labels → ENCODE class codes
UCSC_LABEL_ALIASES = {
    "prom": "PLS",
    "enhP": "pELS",
    "enhD": "dELS",
    "K4m3": "DNase-H3K4me3",
    "CTCF": "CTCF-only",
}

def ccre_meaning(it, rgb):
    cls = next((str(it[f]) for f in CLASS_FIELDS if it.get(f)), None) or CCRE_COLOR_CLASS.get(rgb)
    if not cls:
        unmapped_colors.add(rgb)
        return ""
    cls = UCSC_LABEL_ALIASES.get(cls, cls)
    return f"{cls}: {CCRE_CLASS_TEXT.get(cls, cls)}"

def write_bed(items, chrom, path, name, describe=None):
    lines = [f'track name="{name}" itemRgb="On"']
    for it in items:
        s, e = it["chromStart"], it["chromEnd"]
        rgb = it.get("reserved") or it.get("itemRgb") or "0"
        label = it.get("name", "")
        if describe:
            label = f"{label} {describe(it, rgb)}".strip()
        lines.append(f"{chrom}\t{s}\t{e}\t{label}\t0\t.\t{s}\t{e}\t{rgb}")
    path.write_text("\n".join(lines) + "\n", newline="\n")

In [11]:
OUT_DIR2 = PROJECT_ROOT / "tracks" / "new_data"
write_bed(ucsc_items("encodeCcreCombined", chrom, start1, end1), chrom,
          OUT_DIR2 / "encode3_ccre.bed", "ENCODE3", describe=ccre_meaning)
write_bed(ucsc_items("cCREregistry", chrom, start1, end1), chrom,
          OUT_DIR2 / "encode4_ccre.bed", "ENCODE4", describe=ccre_meaning)
print("unmapped colors:", unmapped_colors or "none")

unmapped colors: none
